In [17]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm



In [18]:
movies = pd.read_csv(r".\lesson36_data\movies_metadata.csv", low_memory=False)

movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

def parse_genres_fast(x):
    if pd.isna(x) or x == '[]':
        return []
    return re.findall(r"'name': '([^']+)'", x)

movies['genres'] = movies['genres'].apply(parse_genres_fast)
movies['overview'] = movies['overview'].fillna('')
movies['tagline'] = movies['tagline'].fillna('')

if 'runtime' in movies.columns:
    movies['runtime'] = pd.to_numeric(movies['runtime'], errors='coerce')
    movies['runtime'] = movies['runtime'].fillna(movies['runtime'].median())
else:
    movies['runtime'] = 90.0

movies['popularity'] = pd.to_numeric(movies['popularity'], errors='coerce').fillna(0)
movies['vote_average'] = pd.to_numeric(movies['vote_average'], errors='coerce').fillna(0)

movies = movies[['id', 'title', 'overview', 'tagline', 'genres', 'popularity', 'vote_average', 'runtime']].copy()

ratings = pd.read_csv(r".\lesson36_data\ratings.csv")
movie_ids = set(movies['id'])
ratings = ratings[ratings['movieId'].isin(movie_ids)].copy()
ratings['liked'] = (ratings['rating'] >= 3.5).astype(int)

train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42, stratify=ratings['liked'])

In [19]:
text_data = (movies['overview'] + ' ' + movies['tagline']).values
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
text_tfidf = tfidf.fit_transform(text_data)

svd_text = TruncatedSVD(n_components=64, random_state=42)
text_features = svd_text.fit_transform(text_tfidf)

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genres'])

svd_genre = TruncatedSVD(n_components=20, random_state=42)
genre_features = svd_genre.fit_transform(genre_matrix)

numeric_data = movies[['popularity', 'vote_average', 'runtime']].values
scaler = StandardScaler()
numeric_features = scaler.fit_transform(numeric_data)

item_features = np.hstack([text_features, genre_features, numeric_features])
movie_map = {mid: i for i, mid in enumerate(movies['id'])}

In [20]:
train_data = train_data.copy()
train_data['movie_idx'] = train_data['movieId'].map(movie_map)
train_clean = train_data.dropna(subset=['movie_idx']).copy()
train_clean['movie_idx'] = train_clean['movie_idx'].astype(int)

users = train_clean['userId'].unique()
user_map = {uid: i for i, uid in enumerate(users)}
train_clean['user_idx'] = train_clean['userId'].map(user_map)

rating_matrix = csr_matrix(
    (train_clean['rating'].values,
     (train_clean['user_idx'].values, train_clean['movie_idx'].values)),
    shape=(len(users), len(movies))
)

user_profiles = rating_matrix @ item_features
sums = np.array(rating_matrix.sum(axis=1)).flatten()
sums[sums == 0] = 1
user_profiles = user_profiles / sums[:, np.newaxis]

In [21]:
def batch_generator(df, user_profiles, item_features, batch_size=50000):
    df_subset = df.dropna(subset=['userId', 'movieId']).copy()
    df_subset['user_idx'] = df_subset['userId'].map(user_map)
    df_subset['movie_idx'] = df_subset['movieId'].map(movie_map)
    
    valid_mask = df_subset['user_idx'].notna() & df_subset['movie_idx'].notna()
    df_subset = df_subset[valid_mask]
    
    user_indices = df_subset['user_idx'].astype(int).values
    movie_indices = df_subset['movie_idx'].astype(int).values
    y_values = df_subset['liked'].values
    
    n_samples = len(df_subset)
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    
    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        batch_idx = indices[start:end]
        
        u_vec = user_profiles[user_indices[batch_idx]]
        i_vec = item_features[movie_indices[batch_idx]]
        
        X_batch = np.hstack([u_vec, i_vec, u_vec * i_vec])
        y_batch = y_values[batch_idx]
        
        yield X_batch, y_batch

In [22]:
model = SGDClassifier(loss='log_loss', max_iter=1, learning_rate='optimal', random_state=42)
epochs = 10
batch_size = 500000

for e in range(epochs):
    train_gen = batch_generator(train_data, user_profiles, item_features, batch_size=batch_size)
    est_batches = int(np.ceil(len(train_data) / batch_size))
    
    for X_batch, y_batch in tqdm(train_gen, total=est_batches, desc=f"Epoch {e+1}"):
        model.partial_fit(X_batch, y_batch, classes=[0, 1])
    
    test_preds = []
    test_truth = []
    test_gen = batch_generator(test_data, user_profiles, item_features, batch_size=batch_size)
    for X_test, y_test in test_gen:
        preds = model.predict_proba(X_test)[:, 1]
        test_preds.extend(preds)
        test_truth.extend(y_test)
        
    roc = roc_auc_score(test_truth, test_preds)
    print(f"Epoch {e+1} ROC-AUC: {roc:.4f}")

Epoch 1: 100%|██████████| 19/19 [00:42<00:00,  2.22s/it]


Epoch 1 ROC-AUC: 0.5955


Epoch 2: 100%|██████████| 19/19 [00:40<00:00,  2.13s/it]


Epoch 2 ROC-AUC: 0.5952


Epoch 3: 100%|██████████| 19/19 [00:39<00:00,  2.07s/it]


...

Epoch 8 ROC-AUC: 0.5959


Epoch 9: 100%|██████████| 19/19 [00:45<00:00,  2.40s/it]


Epoch 9 ROC-AUC: 0.5959


Epoch 10: 100%|██████████| 19/19 [00:38<00:00,  2.02s/it]


Epoch 10 ROC-AUC: 0.5959


In [23]:
for uid in range(1, 6):
    if uid not in user_map:
        continue
        
    u_idx = user_map[uid]
    u_vec = user_profiles[u_idx]
    
    seen = set(ratings[ratings['userId'] == uid]['movieId'])
    unseen_mask = ~movies['id'].isin(seen)
    unseen_idx = np.where(unseen_mask)[0]
    items = item_features[unseen_idx]
    
    scores = []
    batch_inf = 10000
    for start in range(0, len(items), batch_inf):
        end = min(start + batch_inf, len(items))
        i_batch = items[start:end]
        u_batch = np.tile(u_vec, (len(i_batch), 1))
        X_cand = np.hstack([u_batch, i_batch, u_batch * i_batch])
        scores.extend(model.predict_proba(X_cand)[:, 1])
        
    scores = np.array(scores)
    top_indices = np.argsort(scores)[-10:][::-1]
    
    print(f"User {uid} Recommendations:")
    for i, idx in enumerate(top_indices, 1):
        m = movies.iloc[unseen_idx[idx]]
        g = ', '.join(m['genres'])
        print(f"{i}. {m['title']} ({scores[idx]:.4f})\n   Genres: {g}")

User 1 Recommendations:
1. Poil de Carotte (0.9006)
   Genres: Drama, Comedy
2. Piter FM (0.8881)
   Genres: Comedy, Drama, Romance
3. Winter Cherries (0.8847)
   Genres: Comedy, Romance, Drama
4. Hard Choices (0.8773)
   Genres: Drama, Crime
5. Crazy Horse (0.8675)
   Genres: TV Movie, War, Action, Drama, Western
6. Susie Q (0.8630)
   Genres: Comedy, Drama, Family, Mystery, TV Movie
7. Kick-heart (0.8626)
   Genres: Animation, Comedy, Romance
8. My Bromance (0.8619)
   Genres: Comedy, Drama
9. Duel of Hearts (0.8608)
   Genres: TV Movie, Drama, Mystery, Romance
10. Endgame (0.8606)
   Genres: Comedy, Drama, TV Movie
User 2 Recommendations:
1. Minions (0.9306)
   Genres: Family, Animation, Adventure, Comedy
2. Wonder Woman (0.8020)
   Genres: Action, Adventure, Fantasy
3. Beauty and the Beast (0.7603)
   Genres: Family, Fantasy, Romance
4. Guardians of the Galaxy Vol. 2 (0.7575)
   Genres: Action, Adventure, Comedy, Science Fiction
5. Big Hero 6 (0.7522)
   Genres: Adventure, Family, 